# MCTS Foul Play -> Policy MLP Training

This notebook runs the full policy-predictor workflow: clone/open `showdown-trainer`, install dependencies, clone/install Foul Play, collect MCTS policy targets into JSONL, run the repo sanity checks, train from `configs/student.yaml`, and verify that the checkpoint contains both model weights and optimizer state.

The notebook trains one supervised policy predictor. The target is the Foul Play MCTS action distribution; there are no rollout or auxiliary losses.

## 1. Configure Repositories And Run Profile

If this notebook is already inside a local clone, the setup cell reuses it. Otherwise, set `TRAINER_REPO_URL` to your fork and it will clone the repo. Foul Play is cloned separately because it owns the live Showdown/MCTS battle loop.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

TRAINER_REPO_URL = "https://github.com/YOUR_USERNAME/showdown-trainer.git"  # <-- edit this
TRAINER_REPO_BRANCH = "main"
FOUL_PLAY_REPO_URL = "https://github.com/pmariglia/foul-play.git"
FOUL_PLAY_REPO_BRANCH = "main"

CONFIG_PATH = "configs/student.yaml"
SMOKE_RUN = False          # True uses smoke training settings from configs/student.yaml.
COLLECT_REAL_MCTS = True   # False skips live Showdown collection and expects DATA_PATH to already exist.
INSTALL_RUST_IF_MISSING = True


def run(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=True)


def find_existing_repo(start: Path):
    for path in [start, *start.parents]:
        if (path / "train.py").exists() and (path / "dataset.py").exists() and (path / "configs" / "student.yaml").exists():
            return path
    return None


def default_base_dir():
    content = Path("/content")
    if content.exists() and os.access(content, os.W_OK):
        return content
    return Path.home()


base_dir = default_base_dir()
repo = find_existing_repo(Path.cwd())
if repo is None:
    repo = base_dir / "showdown-trainer"
    if not (repo / ".git").exists():
        run(["git", "clone", "--branch", TRAINER_REPO_BRANCH, TRAINER_REPO_URL, repo])
    else:
        print(f"Using existing trainer repo at {repo}")

foul_play_dir = base_dir / "foul-play"
if not foul_play_dir.exists():
    run(["git", "clone", "--branch", FOUL_PLAY_REPO_BRANCH, FOUL_PLAY_REPO_URL, foul_play_dir])
else:
    print(f"Using existing Foul Play repo at {foul_play_dir}")

TRAINER_REPO_DIR = repo.resolve()
FOUL_PLAY_DIR = foul_play_dir.resolve()
print("TRAINER_REPO_DIR =", TRAINER_REPO_DIR)
print("FOUL_PLAY_DIR =", FOUL_PLAY_DIR)

## 2. Install Dependencies

Foul Play uses `poke-engine` for MCTS. Local installs may need Rust available while `poke-engine` builds.

In [ ]:
%cd {TRAINER_REPO_DIR}
!{sys.executable} -m pip install -q -U pip wheel "setuptools<82"

import os
import shutil
import subprocess
from pathlib import Path

if INSTALL_RUST_IF_MISSING and shutil.which("rustc") is None:
    print("Installing Rust toolchain for poke-engine build...")
    subprocess.run("curl https://sh.rustup.rs -sSf | sh -s -- -y", shell=True, check=True)
    cargo_bin = Path.home() / ".cargo" / "bin"
    os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"

!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q -r {FOUL_PLAY_DIR / "requirements.txt"}

import torch
import yaml
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("config:", CONFIG_PATH)

## 3. Configure Pokemon Showdown Credentials

Set these as notebook environment variables or Colab secrets before collecting. `PS_PASSWORD` can be left blank for accounts that do not need a password in your workflow.

In [ ]:
import getpass
import os

if COLLECT_REAL_MCTS:
    if not os.environ.get("PS_USERNAME"):
        os.environ["PS_USERNAME"] = input("Pokemon Showdown username: ").strip()
    if "PS_PASSWORD" not in os.environ:
        password = getpass.getpass("Pokemon Showdown password (blank if none): ")
        os.environ["PS_PASSWORD"] = password
    print("PS_USERNAME set:", bool(os.environ.get("PS_USERNAME")))
else:
    print("Skipping live credential setup because COLLECT_REAL_MCTS = False")

## 4. Collect Foul Play MCTS Training Data

This cell writes `training_data/mcts_run.jsonl` using the `collection` section in `configs/student.yaml`. Adjust `collection.run_count`, `collection.search_time_ms`, `collection.search_parallelism`, and `collection.pokemon_format` in the config for longer data collection.

In [ ]:
from pathlib import Path
import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
DATA_PATH = Path(cfg["training"]["data_path"])
CHECKPOINT_PATH = Path(cfg["training"]["output_path"])

if COLLECT_REAL_MCTS:
    !{sys.executable} collect_foul_play.py --config {CONFIG_PATH} --foul-play-dir {FOUL_PLAY_DIR}
else:
    assert DATA_PATH.exists(), f"{DATA_PATH} does not exist. Enable COLLECT_REAL_MCTS or provide a JSONL dataset."

print("dataset:", DATA_PATH, "exists:", DATA_PATH.exists(), "bytes:", DATA_PATH.stat().st_size if DATA_PATH.exists() else 0)

## 5. Run Tests

The sanity check covers mapping, serialization, encoding, JSONL writing, checkpointing, resume, and masked prediction.

In [ ]:
!{sys.executable} sanity_check.py

## 6. Train From Config

The config controls model dimensions, encoder hash buckets, batch size, learning rate, validation split, gradient clipping, epoch/update budget, and checkpoint path. Checkpoints save both `model_state_dict` and `optimizer_state_dict`, so rerunning with `training.resume_checkpoint_path` set continues optimizer state too.

In [ ]:
smoke_flag = "--smoke" if SMOKE_RUN else ""
!{sys.executable} train.py --config {CONFIG_PATH} {smoke_flag}

## 7. Inspect Checkpoint And One Prediction

In [ ]:
from dataset import iter_decision_records
from encode import EncoderConfig
from train import load_policy_checkpoint, predict_policy
import torch

model, checkpoint = load_policy_checkpoint(CHECKPOINT_PATH)
assert "model_state_dict" in checkpoint
assert "optimizer_state_dict" in checkpoint

first_record = next(iter_decision_records(DATA_PATH))
encoder_config = EncoderConfig(**checkpoint["encoder_config"])
probs = predict_policy(model, first_record["state"], encoder_config=encoder_config)

print("checkpoint epoch:", checkpoint.get("epoch"))
print("checkpoint update:", checkpoint.get("update"))
print("model:", checkpoint["model"])
print("optimizer state groups:", len(checkpoint["optimizer_state_dict"].get("param_groups", [])))
print("policy probs:", [round(p, 4) for p in probs])
print("sum:", round(sum(probs), 6))